In [ ]:
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import torch
import requests
import numpy as np
import os 
import glob
from utils import * 

# Load model and processor
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
model = AutoModel.from_pretrained('facebook/dinov2-base')
model.eval()

# Load image
IMG_DIR = '/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_processed/Dataset999_AutoPet/nnUNetPlans_3d_fullres/'
# print(os.path.exists(IMG_DIR))  # Check if the directory exists
# image_files = glob.glob(os.path.join(IMG_DIR, '*[0-9].nii.gz'))  # Adjust the extension if needed
# image = np.load(image_files[0])  # Load the first image for testing

# Running DinoV2

In [ ]:
# # Handle multi-dimensional medical image
# print(f"Original image shape: {image.shape}")

# # Extract appropriate slice based on dimensionality
# if image.ndim == 4:  # (batch/modality, depth, height, width)
#     img = image[0, image.shape[1]//2]  # Take first modality, middle slice
# elif image.ndim == 3:  # (depth, height, width)
#     img = image[image.shape[0]//2]  # Take middle slice
# else:
#     img = image

# print(f"Extracted slice shape: {img.shape}")

# # Normalize to 0-255 range
# img_norm = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)

# # Convert to PIL Image, then to RGB
# from PIL import Image as PILImage
# pil_img = PILImage.fromarray(img_norm, mode='L').convert('RGB')

# print(f"PIL image size: {pil_img.size}")

# # Process and get embeddings
# inputs = processor(images=pil_img, return_tensors="pt")
# print(f"Input tensor shape: {inputs['pixel_values'].shape}")

# with torch.no_grad():
#     outputs = model(**inputs)

# # Different embedding options:
# # 1. CLS token embedding (most common - single vector per image)
# cls_embedding = outputs.last_hidden_state[:, 0, :]  # shape: [1, 768]

# # 2. Mean of all patch embeddings
# patch_embeddings = outputs.last_hidden_state[:, 1:, :]  # exclude CLS token
# mean_embedding = patch_embeddings.mean(dim=1)  # shape: [1, 768]

# # 3. Pooled output (same as CLS token)
# pooled_embedding = outputs.pooler_output  # shape: [1, 768]

# print(f"\nEmbedding shapes:")
# print(f"  CLS embedding: {cls_embedding.shape}")
# print(f"  Mean embedding: {mean_embedding.shape}")
# print(f"  Patch embeddings: {patch_embeddings.shape}")

# print(f"\nEmbedding statistics:")
# print(f"  CLS - mean: {cls_embedding.mean():.4f}, std: {cls_embedding.std():.4f}")
# print(f"  Mean - mean: {mean_embedding.mean():.4f}, std: {mean_embedding.std():.4f}")

In [ ]:
# # Batch process all images and extract embeddings
# import pandas as pd
# from tqdm import tqdm

# print(f"Found {len(image_files)} images to process\n")

# embeddings_list = []
# metadata_list = []

# for img_path in tqdm(image_files):  # Process first 10 for testing
#     try:
#         # Load image
#         img_data = np.load(img_path)
        
#         # convert to 2D via a mean projection
#         if img_data.ndim == 4:
#             img_2d = img_data[0].mean(axis=0)  # Mean across modalities

#             # normalize to 0-255
#             img_2d = ((img_2d - img_2d.min()) / (img_2d.max() - img_2d.min()) * 255).astype(np.uint8)
            
#         elif img_data.ndim == 3:
#             img_2d = img_data.mean(axis=0)  # Mean across depth
#         else:
#             img_2d = img_data
        
#         # Normalize and convert to PIL RGB
#         pil_img = PILImage.fromarray(img_2d, mode='L').convert('RGB')
        
#         # Get embeddings
#         inputs = processor(images=pil_img, return_tensors="pt")
#         with torch.no_grad():
#             outputs = model(**inputs)
        
#         # Extract CLS token embedding
#         cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
        
#         # Calculate patch statistics
#         patch_emb = outputs.last_hidden_state[:, 1:, :].cpu().numpy()
#         patch_mean = patch_emb.mean(axis=1).flatten()
#         patch_std = patch_emb.std(axis=1).flatten()
        
#         embeddings_list.append({
#             'cls': cls_embedding,
#             'patch_mean': patch_mean,
#             'patch_std': patch_std
#         })
        
#         metadata_list.append({
#             'filename': os.path.basename(img_path),
#             'shape': img_data.shape,
#             'img_min': float(img_2d.min()),
#             'img_max': float(img_2d.max()),
#             'cls_mean': float(cls_embedding.mean()),
#             'cls_std': float(cls_embedding.std())
#         })
        
#     except Exception as e:
#         print(f"Error processing {img_path}: {e}")

# # Create metadata DataFrame
# metadata_df = pd.DataFrame(metadata_list)
# print(f"\nProcessed {len(metadata_df)} images successfully")
# print(f"\nMetadata summary:")
# print(metadata_df.head())
# print(f"\nEmbedding statistics across batch:")
# print(f"CLS embedding - mean: {metadata_df['cls_mean'].mean():.4f}, std: {metadata_df['cls_std'].mean():.4f}")

# Analyzing Results

In [ ]:
import glob
import os
import json 

IMG_DIR = '/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_processed/Dataset999_AutoPet/nnUNetPlans_3d_fullres/'
print(os.path.exists(IMG_DIR))  # Check if the directory exists
SPLIT_JSON = '/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_processed/Dataset999_AutoPet/splits_final.json'

with open(SPLIT_JSON, 'r') as f:
    splits = json.load(f)

    


In [ ]:
import pandas as pd

autopet_embedding_df = pd.read_csv("embedding_normalized__niigz_metadata.csv")
import pickle 

with open("embeddings_normalized__niigz_.pkl", "rb") as f:
    class_embeddings = pickle.load(f)

LABELLED_SPLIT = 0

class_embeddings['filename'] = class_embeddings['filename'].apply(lambda x: x.replace('_0001.nii.gz', ''))  # Remove .nii.gz extension for matching
labelled_class_embeddings = class_embeddings.loc[class_embeddings['filename'].replace('_0001.nii.gz', '').isin(splits[LABELLED_SPLIT]['train'])]
unlabelled_class_embeddings = class_embeddings.loc[~class_embeddings['filename'].replace('_0001.nii.gz', '').isin(splits[LABELLED_SPLIT]['train'])]
validation_class_embeddings = class_embeddings.loc[class_embeddings['filename'].replace('_0001.nii.gz', '').isin(splits[LABELLED_SPLIT]['val'])]




In [ ]:
len(unlabelled_class_embeddings)

In [ ]:
# uncertainty_files_1= pd.read_csv("../nnUNet/bad_uncertainty_files.csv")
# uncertainty_files_2= pd.read_csv("../nnUNet/all_quartiles_uncertainty_cases.csv")

# all_uncertainty_files = pd.concat([uncertainty_files_1, uncertainty_files_2], ignore_index=True)
# all_uncertainty_files['filename'] = all_uncertainty_files['filename'].apply(lambda x: x.replace('.nii.gz', ''))  # Remove .nii.gz extension for matching
# all_uncertainty_files['tracer'] = all_uncertainty_files['filename'].apply(get_tracer_type)


In [ ]:
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
# Extract tracer type from filename
def get_tracer_type(filename):
    """Extract tracer type (FDG or PSMA) from filename"""
    if 'target' in filename.lower():
        return 'PSMA'
    if 'fdg' in filename.lower():
        return 'FDG'
    elif 'psma' in filename.lower():
        return 'PSMA'
    else:
        return 'Unknown'

# Prepare labelled embeddings with tracer info
labelled_cls = np.array([np.array(e) for e in labelled_class_embeddings['cls'].values])
labelled_files = labelled_class_embeddings['filename'].values
labelled_tracers = np.array([get_tracer_type(f) for f in labelled_files])

# Prepare unlabelled embeddings with tracer info
unlabelled_cls = np.array([np.array(e) for e in unlabelled_class_embeddings['cls'].values])
unlabelled_files = unlabelled_class_embeddings['filename'].values
unlabelled_tracers = np.array([get_tracer_type(f) for f in unlabelled_files])

In [ ]:
from sklearn.decomposition import KernelPCA

print("=" * 80)
print("LABELLED DATA ANALYSIS")
print("=" * 80)
print(f"Total labelled samples: {len(labelled_cls)}")
print(f"  FDG: {sum(labelled_tracers == 'FDG')}")
print(f"  PSMA: {sum(labelled_tracers == 'PSMA')}")

print("\n" + "=" * 80)
print("UNLABELLED DATA ANALYSIS")
print("=" * 80)
print(f"Total unlabelled samples: {len(unlabelled_cls)}")
print(f"  FDG: {sum(unlabelled_tracers == 'FDG')}")
print(f"  PSMA: {sum(unlabelled_tracers == 'PSMA')}")

labelled_results = {}
for tracer in ['FDG', 'PSMA', 'DHMC']:
    tracer_mask = labelled_tracers == tracer
    tracer_embeddings = labelled_cls[tracer_mask]
    tracer_files = labelled_files[tracer_mask]
    if len(tracer_embeddings) < 2:
        print(f"\n{tracer}: Only {len(tracer_embeddings)} samples, skipping...")
        continue

    print(f"\n--- {tracer} (n={len(tracer_embeddings)}) ---")
    kpca = KernelPCA(n_components=2, kernel='cosine')
    tracer_kpca = kpca.fit_transform(tracer_embeddings)
    stats = compute_pairwise_cosine_stats(tracer_embeddings, tracer_files)

    print(f"  Kernel PCA (cosine) fitted — 2 components for visualization")
    print(f"  CLOSEST PAIR (redundancy):")
    print(f"    Cosine Distance: {stats['closest'][1]:.6f}")
    print(f"    {stats['closest'][2]}")
    print(f"    {stats['closest'][3]}")
    print(f"  FURTHEST PAIR (diversity):")
    print(f"    Cosine Distance: {stats['furthest'][1]:.6f}")
    print(f"    {stats['furthest'][2]}")
    print(f"    {stats['furthest'][3]}")

    labelled_results[tracer] = {
        'kpca': kpca,
        'kpca_coords': tracer_kpca,
        'embeddings': tracer_embeddings,
        'files': tracer_files,
        'pairwise_distances': stats['pairwise_distances'],
        'closest': stats['closest'],
        'furthest': stats['furthest'],
    }

print("\n" + "=" * 80)
print("UNLABELLED DATA - KERNEL PCA & ACTIVE LEARNING EVALUATION")
print("=" * 80)

unlabelled_results = {}
for tracer in ['FDG', 'PSMA', 'DHMC']:
    tracer_mask_u = unlabelled_tracers == tracer
    tracer_embeddings_u = unlabelled_cls[tracer_mask_u]
    tracer_files_u = unlabelled_files[tracer_mask_u]

    tracer_mask_l = labelled_tracers == tracer
    tracer_embeddings_l = labelled_cls[tracer_mask_l]
    tracer_files_l = labelled_files[tracer_mask_l]

    if len(tracer_embeddings_u) < 2:
        print(f"\n{tracer}: Only {len(tracer_embeddings_u)} unlabelled samples, skipping...")
        continue

    print(f"\n--- {tracer} (unlabelled n={len(tracer_embeddings_u)}, labelled n={len(tracer_embeddings_l)}) ---")
    if len(tracer_embeddings_l) > 0:
        combined_embeddings = np.vstack([tracer_embeddings_u, tracer_embeddings_l])
        kpca = KernelPCA(n_components=2, kernel='cosine')
        combined_kpca = kpca.fit_transform(combined_embeddings)
        unlabelled_kpca = combined_kpca[:len(tracer_embeddings_u)]
        labelled_kpca = combined_kpca[len(tracer_embeddings_u):]
    else:
        kpca = KernelPCA(n_components=2, kernel='cosine')
        unlabelled_kpca = kpca.fit_transform(tracer_embeddings_u)
        labelled_kpca = np.empty((0, 2))

    print(f"  Kernel PCA (cosine) fitted on combined data — 2 components for visualization")

    stats_u = compute_pairwise_cosine_stats(tracer_embeddings_u, tracer_files_u)
    print(f"  CLOSEST UNLABELLED PAIR (redundancy - candidates to remove):")
    print(f"    Cosine Distance: {stats_u['closest'][1]:.6f}")
    print(f"    {stats_u['closest'][2]}")
    print(f"    {stats_u['closest'][3]}")
    print(f"  FURTHEST UNLABELLED PAIR (diversity - important to label):")
    print(f"    Cosine Distance: {stats_u['furthest'][1]:.6f}")
    print(f"    {stats_u['furthest'][2]}")
    print(f"    {stats_u['furthest'][3]}")

    min_dist_to_labelled = compute_unlabelled_distance_to_labelled(tracer_embeddings_u, tracer_embeddings_l)
    k_far = min(5, len(tracer_embeddings_u))
    far_from_labelled_idx = np.argsort(min_dist_to_labelled)[-k_far:][::-1]
    print(f"  TOP-{k_far} FURTHEST FROM LABELLED (novel, priority to label):")
    for rank, fi in enumerate(far_from_labelled_idx):
        print(f"    #{rank+1} cosine_dist={min_dist_to_labelled[fi]:.6f}  {tracer_files_u[fi]}")

    unlabelled_results[tracer] = {
        'kpca': kpca,
        'kpca_coords_unlabelled': unlabelled_kpca,
        'kpca_coords_labelled': labelled_kpca,
        'embeddings_unlabelled': tracer_embeddings_u,
        'embeddings_labelled': tracer_embeddings_l,
        'files_unlabelled': tracer_files_u,
        'files_labelled': tracer_files_l,
        'pairwise_distances_uu': stats_u['pairwise_distances'],
        'min_dist_to_labelled': min_dist_to_labelled,
        'far_from_labelled_idx': far_from_labelled_idx,
        'closest': stats_u['closest'],
        'furthest': stats_u['furthest'],
    }

# Reproducible summary tables and plots



In [ ]:

distance_df = build_distance_dataframe(unlabelled_results)
print("\nDistance dataframe head:")
print(distance_df.head(10).to_string(index=False))

plot_distance_distribution(distance_df)

TOP_PERCENT = 0.01
# Tracer-aware top 5% selection
top_5_df = get_top_percent_farthest_by_tracer(distance_df, top_percent=TOP_PERCENT)
ranked_top_5_df = rank_top_subset_by_diversity(unlabelled_results, top_5_df)

print(f"\n{'='*80}")
print(f"TOP {TOP_PERCENT*100}% FURTHEST IMAGES BY TRACER")
print(f"{'='*80}")
for tracer in ['FDG', 'PSMA', 'DHMC']:
    tracer_top5 = top_5_df[top_5_df['tracer'] == tracer]
    print(f"\n{tracer} (n={len(tracer_top5)})")
    if tracer_top5.empty:
        print("  No samples found.")
    else:
        print(tracer_top5.to_string(index=False))

print(f"\n{'='*80}")
print("TOP 5% RANKED BY DIVERSITY WITHIN THAT SUBSET")
print(f"{'='*80}")
print(ranked_top_5_df.to_string(index=False))

plot_kernel_pca_with_top_subset(unlabelled_results, labelled_results, top_5_df)

print("\n" + "=" * 80)
print("SUMMARY FOR FILE SELECTION")
print("=" * 80)

for tracer in ['FDG', 'PSMA', 'DHMC']:
    if tracer in labelled_results:
        furthest = labelled_results[tracer]['furthest']
        print(f"\nLABELLED {tracer} most diverse pair:")
        print(f"  {furthest[2]}")
        print(f"  {furthest[3]}")

for tracer in ['FDG', 'PSMA', 'DHMC']:
    if tracer in unlabelled_results:
        closest = unlabelled_results[tracer]['closest']
        print(f"\nUNLABELLED {tracer} most redundant pair:")
        print(f"  {closest[2]}")
        print(f"  {closest[3]}")

In [ ]:
top_1_df = get_top_percent_farthest_by_tracer(distance_df, top_percent=0.01)
top_1_df['filename'][0]

In [ ]:
def read_json_splits(json_path):
    with open(json_path, 'r') as f:
        return json.load(f)

maybe_AL_splits = read_json_splits('splits_final.json')

# Initialize new_AL_splits as a dict with string keys (JSON compatible)
new_AL_splits = {}

# Base split (split 0) from the original
new_AL_splits[0] = maybe_AL_splits[0]  # or maybe_AL_splits['0'] depending on your json structure
print(len(new_AL_splits[0]['train']), len(new_AL_splits[0]['val']))

# Initialize empty dicts for each subsequent split
for i in range(1, 9):
    new_AL_splits[i] = {'train': [], 'val': []}

new_AL_splits[1]['train'] = new_AL_splits[0]['train'] + get_top_percent_farthest_by_tracer(distance_df, top_percent=0.01)['filename'].tolist()
new_AL_splits[1]['val']   = new_AL_splits[0]['val']

new_AL_splits[2]['train'] = new_AL_splits[0]['train'] + get_top_percent_farthest_by_tracer(distance_df, top_percent=0.05)['filename'].tolist()
new_AL_splits[2]['val']   = new_AL_splits[0]['val']

new_AL_splits[3]['train'] = new_AL_splits[0]['train'] + get_top_percent_farthest_by_tracer(distance_df, top_percent=0.10)['filename'].tolist()
new_AL_splits[3]['val']   = new_AL_splits[0]['val']

new_AL_splits[4]['train'] = new_AL_splits[0]['train'] + get_top_percent_farthest_by_tracer(distance_df, top_percent=0.20)['filename'].tolist()
new_AL_splits[4]['val']   = new_AL_splits[0]['val']

new_AL_splits[5]['train'] = new_AL_splits[0]['train'] + get_top_percent_farthest_by_tracer(distance_df, top_percent=0.30)['filename'].tolist()
new_AL_splits[5]['val']   = new_AL_splits[0]['val']

new_AL_splits[6]['train'] = new_AL_splits[0]['train'] + get_top_percent_farthest_by_tracer(distance_df, top_percent=0.40)['filename'].tolist()
new_AL_splits[6]['val']   = new_AL_splits[0]['val']

new_AL_splits[7]['train'] = new_AL_splits[0]['train'] + get_top_percent_farthest_by_tracer(distance_df, top_percent=0.50)['filename'].tolist()
new_AL_splits[7]['val']   = new_AL_splits[0]['val']


new_AL_splits[8]['train'] = new_AL_splits[0]['train'] + get_top_percent_farthest_by_tracer(distance_df, top_percent=0.60)['filename'].tolist()
new_AL_splits[8]['val']   = new_AL_splits[0]['val']



# Dump to JSON — convert int keys to strings for JSON compatibility
with open('new_AL_splits.json', 'w') as f:
    json.dump(
        [{'train': v['train'], 'val': v['val']} for v in new_AL_splits.values()],
        f,
        indent=4
    )

In [ ]:
for split in new_AL_splits:
    print(f"Split {split}: Train={len(new_AL_splits[split]['train'])}, Val={len(new_AL_splits[split]['val'])}")

In [ ]:
fdg_unlabelled_df = pd.DataFrame({
    'filename': unlabelled_results['FDG']['files_unlabelled'],
    'min_dist_to_labelled': unlabelled_results['FDG']['min_dist_to_labelled']
}).sort_values('min_dist_to_labelled', ascending=False)

psma_unlabelled_df = pd.DataFrame({
    'filename': unlabelled_results['PSMA']['files_unlabelled'],
    'min_dist_to_labelled': unlabelled_results['PSMA']['min_dist_to_labelled']
}).sort_values('min_dist_to_labelled', ascending=False)

print("\nTop 1 percent unlabelled FDG samples furthest from labelled set:")
print(fdg_unlabelled_df.head(int(len(fdg_unlabelled_df) * 0.01)))

print("\nTop 1 percent unlabelled PSMA samples furthest from labelled set:")
print(psma_unlabelled_df.head(int(len(psma_unlabelled_df) * 0.01)))

In [ ]:
# get all the furthest images sorted by distance from labelled set
print("\n\nUNLABELLED DATA — All images sorted by distance from labelled set (furthest first):")
for tracer in ['FDG', 'PSMA']:  
    if tracer in unlabelled_results:
        res = unlabelled_results[tracer]
        min_dist_to_lab = res['min_dist_to_labelled']
        files = res['files_unlabelled']
        
        # Sort by distance in descending order (furthest first)
        sorted_indices = np.argsort(min_dist_to_lab)[::-1]
        
        print(f"\n  {tracer} (total: {len(files)} samples):")
        for rank, idx in enumerate(sorted_indices, 1):
            dist = min_dist_to_lab[idx]
            filename = files[idx]
            print(f"    #{rank:3d}  dist={dist:.6f}  {filename}")
    else:
        print(f"\n  {tracer}: No unlabelled results found.")


In [ ]:
# Create dataframe and analyze top 5% diverse images
import pandas as pd
import matplotlib.pyplot as plt

all_data = []

for tracer in ['FDG', 'PSMA']:
    if tracer in unlabelled_results:
        res = unlabelled_results[tracer]
        min_dist_to_lab = res['min_dist_to_labelled']
        files = res['files_unlabelled']
        
        for idx, (filename, dist) in enumerate(zip(files, min_dist_to_lab)):
            all_data.append({
                'tracer': tracer,
                'filename': filename,
                'dist_to_labelled': dist,
                'index': idx
            })

# Create dataframe and sort by distance
df = pd.DataFrame(all_data)
df = df.sort_values('dist_to_labelled', ascending=False).reset_index(drop=True)

print(f"Total unlabelled samples: {len(df)}")
print(f"\nDataFrame head:")
print(df.head(10))

# Plot histogram of distance distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall histogram
axes[0].hist(df['dist_to_labelled'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Cosine Distance to Labelled Set', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Distance Distribution - All Unlabelled Data', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Histogram by tracer
for tracer in ['FDG', 'PSMA']:
    tracer_data = df[df['tracer'] == tracer]['dist_to_labelled']
    axes[1].hist(tracer_data, bins=20, alpha=0.6, label=tracer, edgecolor='black')
axes[1].set_xlabel('Cosine Distance to Labelled Set', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distance Distribution - By Tracer', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Get top 5% furthest images
top_5_percent = int(np.ceil(len(df) * 0.05))
top_5_percent_df = df.head(top_5_percent).copy()

print(f"\n{'='*80}")
print(f"TOP 5% FURTHEST IMAGES (n={top_5_percent})")
print(f"{'='*80}")
print(top_5_percent_df.to_string(index=False))

# Among top 5%, find those that are also far from each other
print(f"\n{'='*80}")
print(f"FILTERING TOP 5% FOR DIVERSITY (far from each other)")
print(f"{'='*80}")

for tracer in ['FDG', 'PSMA']:
    tracer_top5 = top_5_percent_df[top_5_percent_df['tracer'] == tracer]
    
    if len(tracer_top5) < 2:
        print(f"\n{tracer}: Only {len(tracer_top5)} samples in top 5%, skipping diversity check.")
        if len(tracer_top5) == 1:
            print(f"  {tracer_top5.iloc[0]['filename']}")
        continue
    
    # Get indices in original unlabelled array
    res = unlabelled_results[tracer]
    orig_indices = tracer_top5['index'].values
    
    # Get embeddings for top 5% samples
    top5_embeddings = res['embeddings_unlabelled'][orig_indices]
    
    # Compute pairwise distances among top 5%
    intra_distances = squareform(pdist(top5_embeddings, metric='cosine'))
    
    # Get average distance to other samples for each point (diversity metric)
    avg_distances = intra_distances.mean(axis=1)
    
    # Sort by diversity
    diversity_order = np.argsort(avg_distances)[::-1]
    
    print(f"\n{tracer} - Top 5% images sorted by diversity (distance to other top 5% samples):")
    for rank, local_idx in enumerate(diversity_order, 1):
        global_idx = tracer_top5.index[local_idx]
        filename = tracer_top5.iloc[local_idx]['filename']
        dist_to_lab = tracer_top5.iloc[local_idx]['dist_to_labelled']
        avg_dist_to_top5 = avg_distances[local_idx]
        print(f"  #{rank}  dist_to_labelled={dist_to_lab:.6f}  avg_dist_to_top5={avg_dist_to_top5:.6f}  {filename}")
